In [1]:
# Import necessary libraries
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the Pima Indian Diabetes dataset from sklearn
# Note: Scikit-learn's built-in 'load_diabetes' is a regression dataset.
# We will load the actual diabetes dataset from an external source
import pandas as pd

# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

# Load the dataset
df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [2]:
import numpy as np

# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())


Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [3]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')


Training set shape: (537, 8)
Test set shape: (231, 8)


In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


In [5]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters


[I 2025-08-10 23:40:07,160] A new study created in memory with name: no-name-f612bb45-2d85-459f-a631-8f8c469097a3
[I 2025-08-10 23:40:07,581] Trial 0 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 169, 'max_depth': 11}. Best is trial 0 with value: 0.7672253258845437.
[I 2025-08-10 23:40:07,714] Trial 1 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 66, 'max_depth': 18}. Best is trial 1 with value: 0.7709497206703911.
[I 2025-08-10 23:40:07,988] Trial 2 finished with value: 0.7746741154562383 and parameters: {'n_estimators': 137, 'max_depth': 15}. Best is trial 2 with value: 0.7746741154562383.
[I 2025-08-10 23:40:08,266] Trial 3 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 147, 'max_depth': 8}. Best is trial 2 with value: 0.7746741154562383.
[I 2025-08-10 23:40:08,615] Trial 4 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 164, 'max_depth': 9}. Best is trial 2 with value: 0.7746741

In [6]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7839851024208566
Best hyperparameters: {'n_estimators': 119, 'max_depth': 15}


In [7]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.75


## Samplers in Optuna

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


In [9]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2025-08-10 23:40:20,331] A new study created in memory with name: no-name-a1ba77ac-6c57-481a-b313-bd19d0ae6520
[I 2025-08-10 23:40:20,712] Trial 0 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 183, 'max_depth': 8}. Best is trial 0 with value: 0.7616387337057727.
[I 2025-08-10 23:40:20,997] Trial 1 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 145, 'max_depth': 16}. Best is trial 1 with value: 0.7709497206703911.
[I 2025-08-10 23:40:21,103] Trial 2 finished with value: 0.7579143389199254 and parameters: {'n_estimators': 55, 'max_depth': 6}. Best is trial 1 with value: 0.7709497206703911.
[I 2025-08-10 23:40:21,363] Trial 3 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 134, 'max_depth': 10}. Best is trial 1 with value: 0.7709497206703911.
[I 2025-08-10 23:40:21,545] Trial 4 finished with value: 0.7541899441340781 and parameters: {'n_estimators': 93, 'max_depth': 9}. Best is trial 1 with value: 0.770949720

In [10]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7783985102420856
Best hyperparameters: {'n_estimators': 134, 'max_depth': 7}


In [11]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.75


In [12]:
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [5, 10, 15, 20]
}

In [13]:
# Create a study and optimize it using GridSampler
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective)

[I 2025-08-10 23:40:32,531] A new study created in memory with name: no-name-b05f6701-56ad-4e56-bd17-17c97e2f0a22
[I 2025-08-10 23:40:32,713] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2025-08-10 23:40:33,008] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2025-08-10 23:40:33,109] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2025-08-10 23:40:33,308] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2025-08-10 23:40:33,505] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.772811

In [14]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7746741154562384
Best hyperparameters: {'n_estimators': 150, 'max_depth': 15}


In [15]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


## Optuna Visualizations

In [16]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [17]:
# 1. Optimization History
plot_optimization_history(study).show()

In [18]:
# 2. Parallel Coordinates Plot
plot_parallel_coordinate(study).show()

In [19]:
# 3. Slice Plot
plot_slice(study).show()

In [20]:
# 4. Contour Plot
plot_contour(study).show()

In [21]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

## Optimizing Multiple ML Models

In [22]:
# Importing the required libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [23]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [24]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2025-08-10 23:40:37,170] A new study created in memory with name: no-name-ced9ace3-0a32-433d-9284-d10d849c1997
[I 2025-08-10 23:40:37,433] Trial 0 finished with value: 0.7728119180633147 and parameters: {'classifier': 'RandomForest', 'n_estimators': 152, 'max_depth': 15, 'min_samples_split': 9, 'min_samples_leaf': 5, 'bootstrap': False}. Best is trial 0 with value: 0.7728119180633147.
[I 2025-08-10 23:40:37,810] Trial 1 finished with value: 0.7635009310986964 and parameters: {'classifier': 'RandomForest', 'n_estimators': 203, 'max_depth': 7, 'min_samples_split': 5, 'min_samples_leaf': 1, 'bootstrap': True}. Best is trial 0 with value: 0.7728119180633147.
[I 2025-08-10 23:40:38,643] Trial 2 finished with value: 0.74487895716946 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 177, 'learning_rate': 0.04616788242991562, 'max_depth': 14, 'min_samples_split': 2, 'min_samples_leaf': 8}. Best is trial 0 with value: 0.7728119180633147.
[I 2025-08-10 23:40:38,653] Trial 3 f

In [25]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.13913066093121446, 'kernel': 'linear', 'gamma': 'auto'}
Best trial accuracy: 0.7895716945996275


In [26]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.772812,2025-08-10 23:40:37.171559,2025-08-10 23:40:37.433784,0 days 00:00:00.262225,NaN,False,RandomForest,NaN,NaN,NaN,15.0,5.0,9.0,152.0,COMPLETE
1,1,0.763501,2025-08-10 23:40:37.434327,2025-08-10 23:40:37.810882,0 days 00:00:00.376555,NaN,True,RandomForest,NaN,NaN,NaN,7.0,1.0,5.0,203.0,COMPLETE
2,2,0.744879,2025-08-10 23:40:37.811336,2025-08-10 23:40:38.643457,0 days 00:00:00.832121,NaN,NaN,GradientBoosting,NaN,NaN,0.046168,14.0,8.0,2.0,177.0,COMPLETE
3,3,0.687151,2025-08-10 23:40:38.643945,2025-08-10 23:40:38.653028,0 days 00:00:00.009083,26.729176,NaN,SVM,scale,sigmoid,NaN,NaN,NaN,NaN,NaN,COMPLETE
4,4,0.754190,2025-08-10 23:40:38.653363,2025-08-10 23:40:39.062356,0 days 00:00:00.408993,NaN,True,RandomForest,NaN,NaN,NaN,19.0,7.0,9.0,232.0,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.716946,2025-08-10 23:40:49.999520,2025-08-10 23:40:50.011474,0 days 00:00:00.011954,0.196743,NaN,SVM,auto,poly,NaN,NaN,NaN,NaN,NaN,COMPLETE
96,96,0.789572,2025-08-10 23:40:50.011920,2025-08-10 23:40:50.022645,0 days 00:00:00.010725,0.117838,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
97,97,0.763501,2025-08-10 23:40:50.023009,2025-08-10 23:40:50.263889,0 days 00:00:00.240880,NaN,NaN,GradientBoosting,NaN,NaN,0.020946,4.0,7.0,5.0,88.0,COMPLETE
98,98,0.767225,2025-08-10 23:40:50.264378,2025-08-10 23:40:50.279934,0 days 00:00:00.015556,0.327176,NaN,SVM,auto,rbf,NaN,NaN,NaN,NaN,NaN,COMPLETE


In [27]:
study.trials_dataframe()['params_classifier'].value_counts()

params_classifier
SVM                 79
RandomForest        11
GradientBoosting    10
Name: count, dtype: int64

In [28]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

params_classifier
GradientBoosting    0.748790
RandomForest        0.765702
SVM                 0.775758
Name: value, dtype: float64

In [29]:
# 1. Optimization History
plot_optimization_history(study).show()

In [30]:
# 3. Slice Plot
plot_slice(study).show()

In [31]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

In [39]:
import optuna
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
import numpy as np

# Load the Iris dataset
X, y = load_iris(return_X_y=True)

# Split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the objective function for XGBoost
def objective(trial):
    # Hyperparameter search space
    param = {
        'verbosity': 0,
        'objective': 'multi:softprob',
        'num_class': 3,
        'eval_metric': 'mlogloss',  # Ensure that the eval_metric is specified here
        'booster': 'gbtree',
        'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-8, 1.0, log=True),
        'eta': trial.suggest_float('eta', 0.01, 0.3),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'n_estimators': 300,
    }

    # Create DMatrix for XGBoost
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)

    # Define a pruning callback based on evaluation metrics
    pruning_callback = optuna.integration.XGBoostPruningCallback(trial, "eval-mlogloss")  # Match the metric name in the evals list

    # Train the model
    bst = xgb.train(
        param,
        dtrain,
        num_boost_round=300,
        evals=[(dtrain, "train"), (dtest, "eval")],  # Ensure the eval datasets and names are specified
        early_stopping_rounds=30,
        callbacks=[pruning_callback]
    )

    # Predict on the test set
    preds = bst.predict(dtest)
    best_preds = [int(np.argmax(line)) for line in preds]

    # Return accuracy as the objective value
    accuracy = accuracy_score(y_test, best_preds)
    return accuracy

# Create a study with pruning
study = optuna.create_study(direction='maximize', pruner=optuna.pruners.SuccessiveHalvingPruner())
study.optimize(objective, n_trials=50)

# Output the best trial
print(f"Best trial: {study.best_trial.params}")
print(f"Best accuracy: {study.best_value}")


[I 2025-08-10 23:46:27,600] A new study created in memory with name: no-name-783963f2-96f7-4fb0-ba04-9deff6717357


[0]	train-mlogloss:0.87893	eval-mlogloss:0.86878
[1]	train-mlogloss:0.72139	eval-mlogloss:0.69638
[2]	train-mlogloss:0.59626	eval-mlogloss:0.56560
[3]	train-mlogloss:0.50175	eval-mlogloss:0.46781
[4]	train-mlogloss:0.42833	eval-mlogloss:0.39046
[5]	train-mlogloss:0.36543	eval-mlogloss:0.32160
[6]	train-mlogloss:0.32181	eval-mlogloss:0.27411
[7]	train-mlogloss:0.28543	eval-mlogloss:0.23171
[8]	train-mlogloss:0.25719	eval-mlogloss:0.19889
[9]	train-mlogloss:0.23790	eval-mlogloss:0.17896
[10]	train-mlogloss:0.22043	eval-mlogloss:0.16054
[11]	train-mlogloss:0.21449	eval-mlogloss:0.15415
[12]	train-mlogloss:0.20107	eval-mlogloss:0.13992
[13]	train-mlogloss:0.19593	eval-mlogloss:0.13297
[14]	train-mlogloss:0.19565	eval-mlogloss:0.13281
[15]	train-mlogloss:0.19161	eval-mlogloss:0.12727
[16]	train-mlogloss:0.19124	eval-mlogloss:0.12557
[17]	train-mlogloss:0.18960	eval-mlogloss:0.12349
[18]	train-mlogloss:0.18948	eval-mlogloss:0.12378
[19]	train-mlogloss:0.18901	eval-mlogloss:0.12361
[20]	train

[I 2025-08-10 23:46:28,096] Trial 0 finished with value: 1.0 and parameters: {'lambda': 8.048819019896068e-08, 'alpha': 0.00011367832445698154, 'eta': 0.1825246479890449, 'gamma': 0.1049692545466821, 'max_depth': 7, 'min_child_weight': 8, 'subsample': 0.8608721497712715, 'colsample_bytree': 0.9218649737220733}. Best is trial 0 with value: 1.0.


[0]	train-mlogloss:0.97132	eval-mlogloss:0.96235
[1]	train-mlogloss:0.83535	eval-mlogloss:0.81490
[2]	train-mlogloss:0.72239	eval-mlogloss:0.69703
[3]	train-mlogloss:0.64975	eval-mlogloss:0.62649
[4]	train-mlogloss:0.57229	eval-mlogloss:0.54515
[5]	train-mlogloss:0.50461	eval-mlogloss:0.47069
[6]	train-mlogloss:0.45971	eval-mlogloss:0.42224
[7]	train-mlogloss:0.41722	eval-mlogloss:0.37433
[8]	train-mlogloss:0.38019	eval-mlogloss:0.32959
[9]	train-mlogloss:0.35277	eval-mlogloss:0.29966
[10]	train-mlogloss:0.32229	eval-mlogloss:0.26529
[11]	train-mlogloss:0.30680	eval-mlogloss:0.24924
[12]	train-mlogloss:0.29210	eval-mlogloss:0.23364
[13]	train-mlogloss:0.27342	eval-mlogloss:0.21184
[14]	train-mlogloss:0.25762	eval-mlogloss:0.19447
[15]	train-mlogloss:0.24564	eval-mlogloss:0.18150
[16]	train-mlogloss:0.24042	eval-mlogloss:0.17344
[17]	train-mlogloss:0.23620	eval-mlogloss:0.16891
[18]	train-mlogloss:0.23516	eval-mlogloss:0.16812
[19]	train-mlogloss:0.23487	eval-mlogloss:0.16802
[20]	train

[I 2025-08-10 23:46:28,499] Trial 1 finished with value: 1.0 and parameters: {'lambda': 3.0350477990974823e-05, 'alpha': 9.984470208004599e-08, 'eta': 0.13370245106481077, 'gamma': 0.14778320454620691, 'max_depth': 8, 'min_child_weight': 9, 'subsample': 0.8559238646261854, 'colsample_bytree': 0.6245170553845631}. Best is trial 0 with value: 1.0.


...

[0]	train-mlogloss:0.86220	eval-mlogloss:0.84133
[1]	train-mlogloss:0.69564	eval-mlogloss:0.66076
[2]	train-mlogloss:0.57045	eval-mlogloss:0.53072


[I 2025-08-10 23:46:33,144] Trial 48 pruned. Trial was pruned at iteration 2.


[0]	train-mlogloss:1.08850	eval-mlogloss:1.08778
[1]	train-mlogloss:1.07560	eval-mlogloss:1.07429
[2]	train-mlogloss:1.06267	eval-mlogloss:1.06085
[3]	train-mlogloss:1.05259	eval-mlogloss:1.05071
[4]	train-mlogloss:1.04035	eval-mlogloss:1.03792
[5]	train-mlogloss:1.02795	eval-mlogloss:1.02464
[6]	train-mlogloss:1.01804	eval-mlogloss:1.01415
[7]	train-mlogloss:1.00667	eval-mlogloss:1.00247
[8]	train-mlogloss:0.99508	eval-mlogloss:0.98998
[9]	train-mlogloss:0.98524	eval-mlogloss:0.97966
[10]	train-mlogloss:0.97414	eval-mlogloss:0.96828
[11]	train-mlogloss:0.96491	eval-mlogloss:0.95870
[12]	train-mlogloss:0.95815	eval-mlogloss:0.95145
[13]	train-mlogloss:0.94739	eval-mlogloss:0.94047
[14]	train-mlogloss:0.93685	eval-mlogloss:0.92910
[15]	train-mlogloss:0.92886	eval-mlogloss:0.92142
[16]	train-mlogloss:0.92089	eval-mlogloss:0.91250
[17]	train-mlogloss:0.91089	eval-mlogloss:0.90197
[18]	train-mlogloss:0.90092	eval-mlogloss:0.89161
[19]	train-mlogloss:0.89336	eval-mlogloss:0.88377
[20]	train

[I 2025-08-10 23:46:33,315] Trial 49 pruned. Trial was pruned at iteration 32.


Best trial: {'lambda': 8.048819019896068e-08, 'alpha': 0.00011367832445698154, 'eta': 0.1825246479890449, 'gamma': 0.1049692545466821, 'max_depth': 7, 'min_child_weight': 8, 'subsample': 0.8608721497712715, 'colsample_bytree': 0.9218649737220733}
Best accuracy: 1.0


In [40]:
from optuna.visualization import plot_intermediate_values

# 1. Plot intermediate values during the trials
plot_intermediate_values(study).show()